In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

In [18]:
words = pd.read_csv('../words_cefr.csv')
texts = pd.read_csv('../cefr_level_texts_with_stats.csv')

In [5]:
words.head(10)

,headword,CEFR
0,a,A1
1,a.m./A.M./am/AM,A1
2,abandon,B1
3,abandoned,B2
4,ability,A2
5,able,B1
6,abnormal,B1
7,abnormally,B2
8,aboard,B1
9,abolish,B2


In [19]:
texts.head()

,text,label,word_count,num_sentences,avg_sentence_length
0,Hi!\nI've been meaning to write for ages and f...,B2,459,27,17.000000
1,﻿It was not so much how hard people found the ...,B2,677,34,19.911765
2,Keith recently came back from a trip to Chicag...,B2,236,14,16.857143
3,"The Griffith Observatory is a planetarium, and...",B2,301,17,17.705882
4,-LRB- The Hollywood Reporter -RRB- It's offici...,B2,352,20,17.600000


In [ ]:
label_encoder = LabelEncoder()
texts["label_encoded"] = label_encoder.fit_transform(texts["label"])

X = texts['text']
y = texts["label_encoded"]

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [23]:
cefr_rank = {
    "A1": 1,
    "A2": 2,
    "B1": 3,
    "B2": 4,
    "C1": 5,
    "C2": 6
}

In [24]:
words["level_num"] = words["CEFR"].map(cefr_rank)

word_level_dict = dict(
    zip(words["headword"], words["level_num"])
)

In [25]:
words.head()

,headword,CEFR,level_num
0,a,A1,1
1,a.m./A.M./am/AM,A1,1
2,abandon,B1,3
3,abandoned,B2,4
4,ability,A2,2


In [26]:
def lexical_cefr_features(texts):
    features = []

    for text in texts:
        words = text.lower().split()
        levels = [
            word_level_dict[w]
            for w in words
            if w in word_level_dict
        ]

        if levels:
            features.append([
                np.mean(levels),      # avg lexical level
                np.max(levels),       # hardest word
                np.std(levels),       # lexical spread
                len(levels)           # known words
            ])
        else:
            features.append([0, 0, 0, 0])

    return np.array(features)

In [34]:
from sklearn.pipeline import FunctionTransformer


lexical_transformer = FunctionTransformer(
    lexical_cefr_features,
    validate=False
)

In [35]:
custom_stopwords = [
    "i", "you", "he", "she", "we", "they",
    "me", "him", "her", "okay", "yes"
]

tfidf = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=8000,
    min_df=3,
    stop_words=custom_stopwords
)

In [36]:
from sklearn.pipeline import FeatureUnion

features = FeatureUnion([
    ("tfidf", tfidf),
    ("lexical_cefr", lexical_transformer)
])

In [60]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("features", features),
    ("clf", LogisticRegression(
        max_iter=6000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

In [61]:
pipeline.fit(X_train, y_train)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('tfidf',
                                                 TfidfVectorizer(max_features=8000,
                                                                 min_df=3,
                                                                 ngram_range=(1,
                                                                              3),
                                                                 stop_words=['i',
                                                                             'you',
                                                                             'he',
                                                                             'she',
                                                                             'we',
                                                                             'they',
                                                                             'me',
                                                                             'him',
                                                                             'her',
                                                                             'okay',
                                                                             'yes'])),
                                                ('lexical_cefr',
                                                 FunctionTransformer(func=<function lexical_cefr_features at 0x320334040>))])),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=6000,
                                    n_jobs=-1))])

In [62]:
from sklearn.metrics import classification_report


y_pred = pipeline.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)

              precision    recall  f1-score   support

          A1       0.74      0.90      0.81        58
          A2       0.73      0.67      0.70        55
          B1       0.62      0.44      0.51        41
          B2       0.55      0.49      0.52        57
          C1       0.46      0.60      0.52        48
          C2       0.74      0.65      0.69        40

    accuracy                           0.64       299
   macro avg       0.64      0.63      0.63       299
weighted avg       0.64      0.64      0.63       299



In [65]:
conf_matrix = confusion_matrix(y_test, y_pred)

labels = ["A1", "A2", "B1", "B2", "C1", "C2"]

df_cm = pd.DataFrame(conf_matrix, index=labels, columns=labels)
print(df_cm)

    A1  A2  B1  B2  C1  C2
A1  52   6   0   0   0   0
A2  14  37   3   1   0   0
B1   3   3  18  13   4   0
B2   1   3   5  28  17   3
C1   0   2   3   8  29   6
C2   0   0   0   1  13  26
